# Week 4 - Q10: Stock-scale investigation
**Question.** Purchase receipts are 18x units sold. Are the PO quantities or the sales quantities on the wrong scale?

Preview built for the team's Q10 (Data Analyst task). It uses only the corrected Week-2 merged files. Every number in the report comes from this notebook, and it stops if any check fails.

In [1]:
import pandas as pd, numpy as np, os, glob
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    """Look in an environment variable first, then in this folder and every parent folder, using the repository layout."""
    if os.environ.get(env_var) and os.path.exists(os.path.join(os.environ[env_var], marker)):
        return os.environ[env_var]
    for base in [os.getcwd()]+[str(p) for p in __import__("pathlib").Path(os.getcwd()).parents]:
        for rel in rels:
            if os.path.exists(os.path.join(base, rel, marker)):
                return os.path.join(base, rel)
    return None
MERGED=find_dir("CADETX_MERGED","sales_orders_full.csv",["merged","data/merged","week-02/reports/week_02/merged","reports/week_02/merged"])
assert MERGED is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
print("merged folder:",MERGED)
os.makedirs("q10_outputs",exist_ok=True); os.makedirs("q10_charts",exist_ok=True)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL"))
    assert ok,f"{name}: expected {expected}, got {actual}"
po=pd.read_csv(f"{MERGED}/purchase_orders_full.csv"); so=pd.read_csv(f"{MERGED}/sales_orders_full.csv",parse_dates=["order_date"])
led=pd.read_csv(f"{MERGED}/stock_ledger_enriched.csv",parse_dates=["movement_date"]); inv=pd.read_csv(f"{MERGED}/inventory_with_ledger.csv")
dl=so[so.order_status=="Delivered"]; rec=po[po.po_status=="Received"]
print(len(po),len(so),len(led),len(inv))

merged folder: /tmp/claude-0/-home-claude/e17ce6b6-afe8-5a4a-b191-c5fe2b31efe5/scratchpad/repo_test/week-02/reports/week_02/merged


155495 130402 237230 180


## Part 0. Integrity checks

In [2]:
check("PO lines",155495,len(po),len(po)==155495)
check("Sales lines",130402,len(so),len(so)==130402)
check("Delivered units tie to Week 3 report",1234939,int(dl.quantity.sum()),int(dl.quantity.sum())==1234939)
check("Received PO units tie to Week 3 report",22387123,int(rec.quantity.sum()),int(rec.quantity.sum())==22387123)
check("Inventory master rows (30 products x 6 branches)",180,len(inv),len(inv)==180)

## Part A. Where the 18x comes from
Units ratio = (lines ratio) x (quantity-per-line ratio).

In [3]:
lines_ratio=len(po)/len(so); qty_ratio=po.quantity.mean()/so.quantity.mean(); units_ratio=rec.quantity.sum()/dl.quantity.sum()
A=pd.DataFrame([["Order lines",len(po),len(so),lines_ratio],["Average quantity per line",po.quantity.mean(),so.quantity.mean(),qty_ratio],
 ["Units (received POs vs delivered orders)",rec.quantity.sum(),dl.quantity.sum(),units_ratio],
 ["Quantity range per line",f"{po.quantity.min()}-{po.quantity.max()}",f"{so.quantity.min()}-{so.quantity.max()}",np.nan]],columns=["measure","purchase_orders","sales","ratio"])
A.to_csv("q10_outputs/q10_gap_decomposition.csv",index=False); print(A.round(2).to_string(index=False))
print("quartiles PO",po.quantity.quantile([.25,.5,.75]).tolist(),"SO",so.quantity.quantile([.25,.5,.75]).tolist())
check("Units ratio is about 18.1",18.1,round(units_ratio,1),abs(units_ratio-18.13)<0.05)
check("Lines ratio x quantity ratio ~ units ratio (all-status, rough)",round(units_ratio,1),round(lines_ratio*qty_ratio,1),abs(lines_ratio*qty_ratio-po.quantity.sum()/so.quantity.sum())<1e-6)

                                 measure purchase_orders      sales  ratio
                             Order lines          155495     130402   1.19
               Average quantity per line      159.926146  10.494732  15.24
Units (received POs vs delivered orders)        22387123    1234939  18.13
                 Quantity range per line          20-300       1-20    NaN
quartiles PO [90.0, 160.0, 230.0] SO [6.0, 10.0, 15.0]


## Part B. What it is not: units, product, price

In [4]:
prod=so.drop_duplicates("product_id").set_index("product_id")
g=pd.DataFrame({"po_units":rec.groupby("product_id").quantity.sum(),"sold_units":dl.groupby("product_id").quantity.sum()}); g["ratio"]=g.po_units/g.sold_units
g["uom"]=prod.uom; g.to_csv("q10_outputs/q10_ratio_by_product.csv")
uom=g.groupby("uom").apply(lambda x:x.po_units.sum()/x.sold_units.sum()).rename("ratio").reset_index(); uom.to_csv("q10_outputs/q10_ratio_by_uom.csv",index=False)
f_po=stats.f_oneway(*[x.quantity.values for _,x in po.groupby("product_id")]).pvalue
f_so=stats.f_oneway(*[x.quantity.values for _,x in so.groupby("product_id")]).pvalue
m=po.merge(prod[["unit_cost"]].rename(columns={"unit_cost":"master_cost"}),left_on="product_id",right_index=True); cr=(m.unit_cost/m.master_cost)
print("product ratio range",g.ratio.min().round(1),g.ratio.max().round(1)); print(uom.round(2).to_string(index=False))
print("ANOVA p PO",round(f_po,3),"SO",round(f_so,3),"| PO unit cost / master cost mean",round(cr.mean(),2),"range",round(cr.min(),2),round(cr.max(),2))
check("Every product ratio between 17 and 20",True,(round(g.ratio.min(),1),round(g.ratio.max(),1)),bool(g.ratio.between(17,20).all()))
check("Ratio by unit of measure within 1 point of each other",True,uom.ratio.round(1).tolist(),float(uom.ratio.max()-uom.ratio.min())<1.0)
check("No product effect on PO quantity (ANOVA p > 0.05)",">0.05",round(f_po,3),f_po>0.05)
check("PO unit cost is on the master-cost scale (mean ratio 0.9-1.2)","0.9-1.2",round(cr.mean(),2),0.9<cr.mean()<1.2)

product ratio range 17.4 19.2
      uom  ratio
cartridge  18.05
    piece  18.08
      set  18.35
ANOVA p PO 0.143 SO 0.395 | PO unit cost / master cost mean 1.06 range 0.72 1.66


## Part C. Which side is right
Independent evidence: stock thresholds, and the cost of goods implied by sales.

In [5]:
yrs=(so.order_date.max()-so.order_date.min()).days/365.25
daily=dl.groupby(["product_id","branch_id"]).quantity.sum()/(yrs*365.25)
iv=inv.merge(daily.rename("daily"),left_on=["product_id","branch_id"],right_index=True)
cov_re=(iv.reorder_level/iv.daily).median(); cov_mx=(iv.max_stock/iv.daily).median()
po_daily=rec.groupby(["product_id","branch_id"]).quantity.sum()/(yrs*365.25); iv=iv.merge(po_daily.rename("po_daily"),left_on=["product_id","branch_id"],right_index=True)
cov_re_po=(iv.reorder_level/iv.po_daily).median()
cogs=(dl.quantity*dl.unit_cost).sum(); spend=rec.line_total.sum(); scaled=spend/units_ratio
C_=pd.DataFrame([["Reorder level, days of delivered sales (median)",cov_re],["Maximum stock, days of delivered sales (median)",cov_mx],["Reorder level, days of PO receipts (median)",cov_re_po],
 ["PO spend (Rs B)",spend/1e9],["Sales cost of goods (Rs B)",cogs/1e9],["PO spend / cost of goods",spend/cogs],["PO spend after dividing quantity by units ratio (Rs B)",scaled/1e9],["Scaled spend as share of cost of goods",scaled/cogs]],columns=["measure","value"])
C_.to_csv("q10_outputs/q10_which_side.csv",index=False); print(C_.round(2).to_string(index=False))
check("Reorder level is 10-40 days of sales",True,round(cov_re,1),10<cov_re<40)
check("Max stock is 60-150 days of sales",True,round(cov_mx,1),60<cov_mx<150)
check("Scaled PO spend within 10% of sales cost of goods",True,round(scaled/cogs,2),abs(scaled/cogs-1)<0.10)

                                               measure  value
       Reorder level, days of delivered sales (median)  20.39
       Maximum stock, days of delivered sales (median)  99.79
           Reorder level, days of PO receipts (median)   1.10
                                       PO spend (Rs B) 312.03
                            Sales cost of goods (Rs B)  18.01
                              PO spend / cost of goods  17.33
PO spend after dividing quantity by units ratio (Rs B)  17.21
                Scaled spend as share of cost of goods   0.96


## Part D. Ledger replay at corrected scale
Divide every IN quantity by k and replay the ledger from the opening stock. Adjustments are treated as additions because all are positive.

In [6]:
ls=led.sort_values(["product_id","branch_id","movement_date","movement_id"]).copy(); ls["sgn"]=ls.movement_type.map({"IN":1,"OUT":-1,"ADJUSTMENT":1})
ix=inv.set_index(["product_id","branch_id"])
def replay(k):
    r=[]
    for (p,b),x in ls.groupby(["product_id","branch_id"]):
        q=np.where(x.movement_type=="IN",x.quantity/k,x.quantity)*x.sgn.values; bal=ix.opening_stock[(p,b)]+np.cumsum(q)
        r.append((bal.mean(),(bal<0).mean(),(bal>ix.max_stock[(p,b)]).mean()))
    r=np.array(r).mean(0); return dict(k=k,avg_balance=r[0],share_days_below_zero=r[1],share_days_above_max=r[2])
D=pd.DataFrame([replay(k) for k in (1,10,15,units_ratio,20)]); D.to_csv("q10_outputs/q10_ledger_replay.csv",index=False); print(D.round(3).to_string(index=False))
check("As-is balance far above max stock (>95% of days)",">0.95",round(D.share_days_above_max[0],2),D.share_days_above_max[0]>0.95)
check("At k=18.1 average balance within 25% of mean max stock",True,(round(D.avg_balance[3]),round(ix.max_stock.mean())),abs(D.avg_balance[3]/ix.max_stock.mean()-1)<0.25)
check("Balance still goes below zero at k=18.1 (replenishment not stock-driven)",">0.05",round(D.share_days_below_zero[3],2),D.share_days_below_zero[3]>0.05)

     k  avg_balance  share_days_below_zero  share_days_above_max
 1.000    53627.311                  0.000                 0.997
10.000     2860.097                  0.000                 0.965
15.000      979.829                  0.047                 0.756
18.128      330.926                  0.284                 0.466
20.000       39.696                  0.403                 0.326


## Part E. Side issues in the ledger

In [7]:
lin=led[led.movement_type=="IN"]; lout=led[led.movement_type=="OUT"]; adj=led[led.movement_type=="ADJUSTMENT"]
E_=pd.DataFrame([["Ledger IN units / received PO units",lin.quantity.sum()/rec.quantity.sum()],["Ledger OUT units / delivered units",lout.quantity.sum()/dl.quantity.sum()],["Ledger IN / OUT",lin.quantity.sum()/lout.quantity.sum()],["Adjustments: min, max",f"{adj.quantity.min()}, {adj.quantity.max()}"]],columns=["measure","value"])
E_.to_csv("q10_outputs/q10_ledger_side_issues.csv",index=False); print(E_.to_string(index=False))
check("Ledger IN/OUT ratio matches PO/sales ratio (within 0.5)",round(units_ratio,1),round(lin.quantity.sum()/lout.quantity.sum(),1),abs(lin.quantity.sum()/lout.quantity.sum()-units_ratio)<0.5)
check("All ledger adjustments are positive",True,int(adj.quantity.min()),adj.quantity.min()>0)

                            measure      value
Ledger IN units / received PO units    0.91114
 Ledger OUT units / delivered units    0.91084
                    Ledger IN / OUT  18.134081
              Adjustments: min, max      5, 50


## Charts

In [8]:
fig,ax=plt.subplots(figsize=(8,4.2),facecolor=SURF); ax.set_facecolor(SURF)
b=np.arange(0,305,10); ax.hist(po.quantity,bins=b,color=BLUE,alpha=.9,label="Purchase-order lines"); ax.hist(so.quantity,bins=b,color=ORANGE,alpha=.9,label="Sales lines")
ax.set_xlabel("Quantity per order line",color=INK2); ax.set_ylabel("Lines",color=INK2); ax.grid(axis="y",color=GRID,lw=.6); [s.set_visible(False) for k_,s in ax.spines.items() if k_ in("top","right")]
ax.set_title(f"A PO line averages {po.quantity.mean():.0f} units, a sales line {so.quantity.mean():.1f} ({qty_ratio:.1f}x)",loc="left",color=INK,fontsize=12); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("q10_charts/q10_01_quantity_per_line.png",dpi=150,facecolor=SURF); plt.close()
fig,ax=plt.subplots(figsize=(8,5),facecolor=SURF); ax.set_facecolor(SURF); gg=g.sort_values("ratio")
ax.barh(gg.index,gg.ratio,color=BLUE,height=.6); ax.axvline(units_ratio,color=ORANGE,lw=1.5); ax.text(units_ratio+.15,0,f"all products {units_ratio:.1f}x",color=INK2,fontsize=9)
ax.set_xlabel("Received PO units / delivered units",color=INK2); ax.grid(axis="x",color=GRID,lw=.6); [s.set_visible(False) for k_,s in ax.spines.items() if k_ in("top","right")]
ax.set_title("The same 17-19x gap appears in every one of the 30 products",loc="left",color=INK,fontsize=12); plt.tight_layout(); plt.savefig("q10_charts/q10_02_ratio_by_product.png",dpi=150,facecolor=SURF); plt.close()

## Validation report

In [9]:
v=pd.DataFrame(checks); v.to_csv("validation_report_week04_q10.csv",index=False); print(v.to_string(index=False)); assert (v.Status=="PASS").all()

                                                              Check_Item Expected             Actual Status
                                                                PO lines   155495             155495   PASS
                                                             Sales lines   130402             130402   PASS
                                    Delivered units tie to Week 3 report  1234939            1234939   PASS
                                  Received PO units tie to Week 3 report 22387123           22387123   PASS
                        Inventory master rows (30 products x 6 branches)      180                180   PASS
                                               Units ratio is about 18.1     18.1               18.1   PASS
          Lines ratio x quantity ratio ~ units ratio (all-status, rough)     18.1               18.2   PASS
                                   Every product ratio between 17 and 20     True       (17.4, 19.2)   PASS
                   Ratio by 